# 02 — EDA: Fiscal Hawkishness Signal

Exploratory analysis of the dictionary-based hawkishness scores produced by `signal/scoring/tfidf_dictionary.py`.  
Reads from `data/interim/speeches_scored.csv` and `data/interim/paragraphs_scored.csv`.

**Sections**
1. Setup & data loading  
2. Coverage — how much of the corpus fires?  
3. Time series — monthly mean score (raw + z-score)  
4. Score distributions — histograms & box plots by president  
5. Term hit analysis — which terms fire, and for whom?  
6. LDA × signal — fiscal topic probability vs score intensity  
7. Summary statistics table  

## 1. Setup & data loading

In [ ]:
%matplotlib inline
import os, re, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'axes.titlesize': 12, 'axes.labelsize': 10})

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

PRES_ORDER  = ['Macri', 'AF', 'Milei']
PRES_COLORS = {'Macri': '#2196F3', 'AF': '#4CAF50', 'Milei': '#FF5722'}
TRANSITIONS = ['2015-12-10', '2019-12-10', '2023-12-10']

# ── Load data ────────────────────────────────────────────────────────────────
speech = pd.read_csv(os.path.join(ROOT, 'data/interim/speeches_scored.csv'), parse_dates=['date'])
speech = speech[speech['president'].isin(PRES_ORDER)].copy()
speech['ym_dt'] = pd.to_datetime(speech['year_month'])
speech['total_hits'] = speech['hawkish_hits_total'] + speech['dovish_hits_total']

para = pd.read_csv(os.path.join(ROOT, 'data/interim/paragraphs_scored.csv'))
para = para[para['president'].isin(PRES_ORDER)].copy()

# ── Dictionary files for term-level analysis ─────────────────────────────────
def load_terms(path):
    def normalise(t):
        t = t.lower()
        t = ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
        return re.sub(r'\s+', ' ', re.sub(r'[^a-z\s]', ' ', t)).strip()
    terms = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                terms.append(normalise(line))
    return sorted(set(terms), key=len, reverse=True)

def term_pattern(term):
    return re.compile(r'\s+'.join(w + r'\w*' for w in term.split()))

DICT_DIR = os.path.join(ROOT, 'signal/dictionaries')
hawkish_terms = load_terms(os.path.join(DICT_DIR, 'hawkish_terms.txt'))
dovish_terms  = load_terms(os.path.join(DICT_DIR, 'dovish_terms.txt'))
hawkish_pats  = [(t, term_pattern(t)) for t in hawkish_terms]
dovish_pats   = [(t, term_pattern(t)) for t in dovish_terms]

def normalise(text):
    text = text.lower()
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z\s]', ' ', text)).strip()

print(f'Speeches : {len(speech):,}')
print(f'Paragraphs: {len(para):,}')
print(f'Hawkish terms: {len(hawkish_terms)} | Dovish terms: {len(dovish_terms)}')

## 2. Coverage — how much of the corpus fires?

In [ ]:
# ── Speech-level hit rates ─────────────────────────────────────────────────
print('=== SPEECH-LEVEL ZERO-HIT RATE ===')
for pres in PRES_ORDER:
    sub = speech[speech['president'] == pres]
    zero = (sub['total_hits'] == 0)
    print(f'  {pres:6s}: {(~zero).sum():3d}/{len(sub):3d} speeches with ≥1 hit '
          f'({(~zero).mean()*100:.1f}%)')

print()
print('=== PARAGRAPH-LEVEL HIT RATE ===')
for pres in PRES_ORDER:
    sub = para[para['president'] == pres]
    h = (sub['hawkish_hits'] > 0).sum()
    d = (sub['dovish_hits'] > 0).sum()
    any_hit = ((sub['hawkish_hits'] + sub['dovish_hits']) > 0).sum()
    print(f'  {pres:6s}: {any_hit:4d}/{len(sub):5d} paragraphs hit '
          f'({any_hit/len(sub)*100:.1f}%)  hawkish={h}  dovish={d}')

# ── Bar chart: % speeches with signal, by president ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

hit_rates = []
for pres in PRES_ORDER:
    sub = speech[speech['president'] == pres]
    hit_rates.append((~(sub['total_hits'] == 0)).mean() * 100)

ax = axes[0]
bars = ax.bar(PRES_ORDER, hit_rates, color=[PRES_COLORS[p] for p in PRES_ORDER], width=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_ylim(0, 110)
ax.set_ylabel('% speeches with ≥1 dictionary hit')
ax.set_title('Speech-level hit rate by president')

# Hawkish vs dovish paragraphs
ax = axes[1]
x = np.arange(len(PRES_ORDER))
w = 0.35
h_rates, d_rates = [], []
for pres in PRES_ORDER:
    sub = para[para['president'] == pres]
    h_rates.append((sub['hawkish_hits'] > 0).mean() * 100)
    d_rates.append((sub['dovish_hits'] > 0).mean() * 100)

b1 = ax.bar(x - w/2, h_rates, w, label='Hawkish', color='#E53935', alpha=0.85)
b2 = ax.bar(x + w/2, d_rates, w, label='Dovish',  color='#1E88E5', alpha=0.85)
ax.bar_label(b1, fmt='%.1f%%', padding=2, fontsize=8)
ax.bar_label(b2, fmt='%.1f%%', padding=2, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(PRES_ORDER)
ax.set_ylabel('% paragraphs with ≥1 hit')
ax.set_title('Paragraph hit rate: hawkish vs dovish')
ax.legend()
ax.set_ylim(0, 30)

plt.tight_layout()
plt.show()

## 3. Time series — monthly mean score

In [ ]:
monthly = (
    speech.groupby(['year_month', 'president'], observed=True)
    [['net_tf_weighted', 'net_tf_weighted_z', 'net_tf_equal', 'hawkish_tf_weighted', 'dovish_tf_weighted']]
    .mean()
    .reset_index()
)
monthly['ym_dt'] = pd.to_datetime(monthly['year_month'])

fig, axes = plt.subplots(3, 1, figsize=(15, 13), sharex=True)

def add_transitions(ax):
    for d in TRANSITIONS:
        ax.axvline(pd.Timestamp(d), color='grey', linewidth=1, linestyle='--', alpha=0.5)
    ax.axhline(0, color='black', linewidth=0.7, linestyle='--', alpha=0.4)

# ── Panel 1: raw net TF weighted ──────────────────────────────────────────
ax = axes[0]
for pres in PRES_ORDER:
    sub = monthly[monthly['president'] == pres].sort_values('ym_dt')
    ax.plot(sub['ym_dt'], sub['net_tf_weighted'],
            label=pres, color=PRES_COLORS[pres], linewidth=1.8)
add_transitions(ax)
ax.set_ylabel('Net TF (raw)')
ax.set_title('Monthly mean hawkishness — raw net TF (fiscal-probability weighted)')
ax.legend(loc='upper left')

# ── Panel 2: z-score normalised ───────────────────────────────────────────
ax = axes[1]
for pres in PRES_ORDER:
    sub = monthly[monthly['president'] == pres].sort_values('ym_dt')
    ax.plot(sub['ym_dt'], sub['net_tf_weighted_z'],
            label=pres, color=PRES_COLORS[pres], linewidth=1.8)
add_transitions(ax)
ax.set_ylabel('Net TF (z-score)')
ax.set_title('Monthly mean hawkishness — z-score normalised (BVAR input series)')
ax.legend(loc='upper left')

# ── Panel 3: hawkish vs dovish components separately ─────────────────────
ax = axes[2]
for pres in PRES_ORDER:
    sub = monthly[monthly['president'] == pres].sort_values('ym_dt')
    ax.plot(sub['ym_dt'], sub['hawkish_tf_weighted'],
            color=PRES_COLORS[pres], linewidth=1.5, linestyle='-',
            label=f'{pres} hawkish')
    ax.plot(sub['ym_dt'], -sub['dovish_tf_weighted'],
            color=PRES_COLORS[pres], linewidth=1.5, linestyle=':',
            label=f'{pres} dovish (neg)')
add_transitions(ax)
ax.set_ylabel('TF score (dovish reflected)')
ax.set_title('Hawkish (solid) vs dovish (dotted, sign-flipped) components')
ax.legend(loc='upper left', fontsize=7, ncol=2)

axes[2].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# ── Rolling 3-month smoothed series ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 5))

for pres in PRES_ORDER:
    sub = monthly[monthly['president'] == pres].sort_values('ym_dt').set_index('ym_dt')
    smoothed = sub['net_tf_weighted_z'].rolling(3, min_periods=1, center=True).mean()
    ax.plot(smoothed.index, smoothed.values,
            label=pres, color=PRES_COLORS[pres], linewidth=2.2)
    # shade area under/over zero
    ax.fill_between(smoothed.index, smoothed.values, 0,
                    where=(smoothed.values > 0),
                    color=PRES_COLORS[pres], alpha=0.12)
    ax.fill_between(smoothed.index, smoothed.values, 0,
                    where=(smoothed.values < 0),
                    color=PRES_COLORS[pres], alpha=0.12)

for d in TRANSITIONS:
    ax.axvline(pd.Timestamp(d), color='grey', linewidth=1.2,
               linestyle='--', alpha=0.6)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_ylabel('Net TF z-score (3-month rolling mean)')
ax.set_title('Fiscal hawkishness signal — 3-month smoothed z-score')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Score distributions — histograms & box plots by president

In [ ]:
# ── Histograms: raw net TF weighted ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for i, pres in enumerate(PRES_ORDER):
    sub = speech[speech['president'] == pres]['net_tf_weighted'].dropna()
    ax = axes[i]
    ax.hist(sub, bins=60, color=PRES_COLORS[pres], alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(sub.mean(), color='black', linewidth=1.5, linestyle='--', label=f'mean={sub.mean():.4f}')
    ax.axvline(sub.median(), color='grey', linewidth=1.2, linestyle=':', label=f'median={sub.median():.4f}')
    ax.axvline(0, color='red', linewidth=0.8, alpha=0.5)
    ax.set_title(f'{pres} — net TF weighted')
    ax.set_xlabel('Score')
    ax.set_ylabel('Speeches')
    ax.legend(fontsize=8)

plt.suptitle('Speech-level raw score distributions by president', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Histograms: z-score normalised ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for i, pres in enumerate(PRES_ORDER):
    sub = speech[speech['president'] == pres]['net_tf_weighted_z'].dropna()
    ax = axes[i]
    ax.hist(sub, bins=60, color=PRES_COLORS[pres], alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(sub.mean(), color='black', linewidth=1.5, linestyle='--', label=f'mean={sub.mean():.2f}')
    ax.axvline(0, color='red', linewidth=0.8, alpha=0.5, label='zero')
    ax.set_title(f'{pres} — z-score')
    ax.set_xlabel('Z-score')
    ax.set_ylabel('Speeches')
    ax.legend(fontsize=8)

plt.suptitle('Speech-level z-score distributions by president', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Box plots: raw + z-score side by side ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, title in [
    (axes[0], 'net_tf_weighted',   'Raw net TF (fiscal-prob weighted)'),
    (axes[1], 'net_tf_weighted_z', 'Z-score normalised'),
]:
    data = [speech[speech['president'] == p][col].dropna().values for p in PRES_ORDER]
    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    medianprops=dict(color='white', linewidth=2),
                    flierprops=dict(marker='o', markersize=3, alpha=0.4))
    for patch, pres in zip(bp['boxes'], PRES_ORDER):
        patch.set_facecolor(PRES_COLORS[pres])
        patch.set_alpha(0.85)
    ax.set_xticks(range(1, len(PRES_ORDER) + 1))
    ax.set_xticklabels(PRES_ORDER)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel('Score')

plt.suptitle('Speech-level score distributions — box plots', fontsize=12)
plt.tight_layout()
plt.show()

# ── Summary stats table ───────────────────────────────────────────────────────
print('\n=== SPEECH-LEVEL SCORE SUMMARY ===')
for col, label in [('net_tf_weighted', 'Raw net TF'), ('net_tf_weighted_z', 'Z-score')]:
    print(f'\n{label}:')
    print(
        speech[speech['president'].isin(PRES_ORDER)]
        .groupby('president')[col]
        .describe()[['mean','std','min','25%','50%','75%','max']]
        .loc[PRES_ORDER]
        .round(5)
        .to_string()
    )

In [ ]:
# ── Non-zero speeches only: scores conditioned on firing ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, pres in enumerate(PRES_ORDER):
    sub = speech[(speech['president'] == pres) & (speech['total_hits'] > 0)]
    ax = axes[i]
    ax.hist(sub['net_tf_weighted'], bins=40,
            color=PRES_COLORS[pres], alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(sub['net_tf_weighted'].mean(), color='black',
               linewidth=1.5, linestyle='--',
               label=f'mean={sub["net_tf_weighted"].mean():.4f}')
    ax.axvline(0, color='red', linewidth=0.8, alpha=0.5)
    ax.set_title(f'{pres} — {len(sub)} speeches with hits')
    ax.set_xlabel('Net TF score')
    ax.set_ylabel('Speeches')
    ax.legend(fontsize=8)

plt.suptitle('Score distribution — non-zero speeches only', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 5. Term hit analysis — which terms fire, and for whom?

In [ ]:
# ── Count hits per term per president ────────────────────────────────────────
def count_hits_by_president(patterns, direction):
    """Return DataFrame: term × president hit counts."""
    records = []
    for pres in PRES_ORDER:
        texts = para[para['president'] == pres]['text_para'].dropna()
        for term, pat in patterns:
            total = sum(len(pat.findall(normalise(str(t)))) for t in texts)
            if total > 0:
                records.append({'term': term, 'president': pres, 'hits': total, 'direction': direction})
    return pd.DataFrame(records)

print('Counting hawkish term hits by president...')
h_hits = count_hits_by_president(hawkish_pats, 'hawkish')
print('Counting dovish term hits by president...')
d_hits = count_hits_by_president(dovish_pats, 'dovish')
all_hits = pd.concat([h_hits, d_hits], ignore_index=True)

print(f'\nTotal hawkish hits across corpus: {h_hits["hits"].sum():,}')
print(f'Total dovish hits across corpus:  {d_hits["hits"].sum():,}')

In [ ]:
# ── Top hawkish terms — stacked bar by president ──────────────────────────────
h_wide = (
    h_hits.pivot_table(index='term', columns='president', values='hits', aggfunc='sum', fill_value=0)
    .reindex(columns=PRES_ORDER, fill_value=0)
)
h_wide['total'] = h_wide.sum(axis=1)
h_top = h_wide.nlargest(20, 'total').drop(columns='total')

fig, ax = plt.subplots(figsize=(13, 6))
bottom = np.zeros(len(h_top))
for pres in PRES_ORDER:
    if pres in h_top.columns:
        vals = h_top[pres].values
        ax.barh(range(len(h_top)), vals, left=bottom,
                color=PRES_COLORS[pres], label=pres, alpha=0.88)
        bottom += vals
ax.set_yticks(range(len(h_top)))
ax.set_yticklabels(h_top.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Total hits across corpus')
ax.set_title('Top 20 hawkish terms — hits by president')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Top dovish terms — stacked bar by president ───────────────────────────────
d_wide = (
    d_hits.pivot_table(index='term', columns='president', values='hits', aggfunc='sum', fill_value=0)
    .reindex(columns=PRES_ORDER, fill_value=0)
)
d_wide['total'] = d_wide.sum(axis=1)
d_top = d_wide.nlargest(20, 'total').drop(columns='total')

fig, ax = plt.subplots(figsize=(13, 6))
bottom = np.zeros(len(d_top))
for pres in PRES_ORDER:
    if pres in d_top.columns:
        vals = d_top[pres].values
        ax.barh(range(len(d_top)), vals, left=bottom,
                color=PRES_COLORS[pres], label=pres, alpha=0.88)
        bottom += vals
ax.set_yticks(range(len(d_top)))
ax.set_yticklabels(d_top.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Total hits across corpus')
ax.set_title('Top 20 dovish terms — hits by president')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── President-specific term breakdown: who owns which terms? ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for i, pres in enumerate(PRES_ORDER):
    ax = axes[i]
    pres_h = h_hits[h_hits['president'] == pres].nlargest(10, 'hits')
    pres_d = d_hits[d_hits['president'] == pres].nlargest(10, 'hits')

    # Hawkish positive, dovish negative
    terms  = list(pres_h['term']) + list(pres_d['term'])
    values = list(pres_h['hits']) + [-v for v in pres_d['hits']]
    colors = ['#E53935'] * len(pres_h) + ['#1E88E5'] * len(pres_d)

    order = np.argsort(values)[::-1]
    ax.barh([terms[o] for o in order], [values[o] for o in order],
            color=[colors[o] for o in order], alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{pres} — top terms\n(red=hawkish, blue=dovish neg)', fontsize=10)
    ax.tick_params(axis='y', labelsize=7.5)

plt.suptitle('Top 10 hawkish & dovish terms per president', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 6. LDA × signal — fiscal topic probability vs score intensity

In [ ]:
# ── Paragraph-level: fiscal_topic_prob vs net_tf ─────────────────────────────
# Only paragraphs with at least one hit
para_hits = para[(para['hawkish_hits'] + para['dovish_hits']) > 0].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for i, pres in enumerate(PRES_ORDER):
    sub = para_hits[para_hits['president'] == pres]
    ax = axes[i]
    ax.scatter(sub['fiscal_topic_prob'], sub['net_tf'],
               c=PRES_COLORS[pres], alpha=0.35, s=12, edgecolors='none')
    # lowess-style binned mean
    sub_s = sub.sort_values('fiscal_topic_prob')
    bins = pd.cut(sub_s['fiscal_topic_prob'], bins=10)
    binned = sub_s.groupby(bins, observed=True)['net_tf'].mean()
    bin_centers = [iv.mid for iv in binned.index]
    ax.plot(bin_centers, binned.values, color='black', linewidth=1.8, label='binned mean')
    ax.axhline(0, color='red', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title(f'{pres} (n={len(sub):,})')
    ax.set_xlabel('Fiscal topic probability')
    if i == 0:
        ax.set_ylabel('Net TF (paragraph)')
    ax.legend(fontsize=8)

plt.suptitle('Paragraph-level: fiscal topic prob vs net TF score (hit paragraphs only)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Speech-level: fiscal_weight_sum vs net_tf_weighted ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, pres in enumerate(PRES_ORDER):
    sub = speech[(speech['president'] == pres) & speech['net_tf_weighted'].notna()]
    ax = axes[i]
    ax.scatter(sub['fiscal_weight_sum'], sub['net_tf_weighted'],
               c=PRES_COLORS[pres], alpha=0.4, s=14, edgecolors='none')
    ax.axhline(0, color='red', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title(f'{pres}')
    ax.set_xlabel('Fiscal weight sum (Σ fiscal_prob × n_tokens)')
    if i == 0:
        ax.set_ylabel('Net TF weighted')

plt.suptitle('Speech-level: fiscal weight sum vs net TF score', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Fiscal paragraph share over time ─────────────────────────────────────────
monthly_fiscal = (
    speech.groupby(['year_month', 'president'], observed=True)
    .agg(fiscal_paras=('n_fiscal_paragraphs', 'sum'),
         total_paras=('n_paragraphs', 'sum'))
    .reset_index()
)
monthly_fiscal['fiscal_share'] = monthly_fiscal['fiscal_paras'] / monthly_fiscal['total_paras'].clip(lower=1)
monthly_fiscal['ym_dt'] = pd.to_datetime(monthly_fiscal['year_month'])

fig, ax = plt.subplots(figsize=(15, 4))
for pres in PRES_ORDER:
    sub = monthly_fiscal[monthly_fiscal['president'] == pres].sort_values('ym_dt')
    ax.plot(sub['ym_dt'], sub['fiscal_share'] * 100,
            label=pres, color=PRES_COLORS[pres], linewidth=1.8)
for d in TRANSITIONS:
    ax.axvline(pd.Timestamp(d), color='grey', linewidth=1, linestyle='--', alpha=0.5)
ax.set_ylabel('% paragraphs classified fiscal (LDA)')
ax.set_title('Monthly share of paragraphs in fiscal LDA topic — by president')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Primary vs equal-weight robustness scatter (all presidents) ───────────────
fig, ax = plt.subplots(figsize=(7, 6))

for pres in PRES_ORDER:
    sub = speech[speech['president'] == pres]
    ax.scatter(sub['net_tf_equal'], sub['net_tf_weighted'],
               c=PRES_COLORS[pres], alpha=0.35, s=14,
               edgecolors='none', label=pres)

lim = max(abs(speech['net_tf_equal'].max()), abs(speech['net_tf_weighted'].max())) * 1.05
ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8, alpha=0.5, label='45° line')
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel('Equal-weight score (robustness check)')
ax.set_ylabel('Fiscal-prob weighted score (primary)')
ax.set_title('Primary vs robustness score')
ax.legend(fontsize=9)

corr = speech['net_tf_equal'].corr(speech['net_tf_weighted'])
ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes,
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

## 7. Summary statistics

In [ ]:
# ── Full summary table ────────────────────────────────────────────────────────
rows = []
for pres in PRES_ORDER:
    sub = speech[speech['president'] == pres]
    hits = sub[sub['total_hits'] > 0]
    rows.append({
        'President':              pres,
        'Speeches':               len(sub),
        'Speeches w/ signal (%)': f"{(sub['total_hits']>0).mean()*100:.1f}",
        'Mean hawkish hits':      f"{sub['hawkish_hits_total'].mean():.2f}",
        'Mean dovish hits':       f"{sub['dovish_hits_total'].mean():.2f}",
        'Mean net TF (raw)':      f"{sub['net_tf_weighted'].mean():.5f}",
        'Std net TF':             f"{sub['net_tf_weighted'].std():.5f}",
        'Mean z-score':           f"{sub['net_tf_weighted_z'].mean():.3f}",
        'Median z-score':         f"{sub['net_tf_weighted_z'].median():.3f}",
    })

summary = pd.DataFrame(rows).set_index('President')
print(summary.T.to_string())

print('\n=== FISCAL PARAGRAPH SHARE ===')
for pres in PRES_ORDER:
    sub = para[para['president'] == pres]
    fiscal = sub['is_fiscal'].sum()
    print(f'  {pres:6s}: {fiscal:4d}/{len(sub):5d} fiscal paragraphs ({fiscal/len(sub)*100:.1f}%)')